# Florida Real Estate Price Prediction — Autoresearch Results

**Dataset:** [Florida Real Estate: Sold Properties Dataset 2026](https://www.kaggle.com/datasets/kanchana1990/florida-real-estate-sold-dataset-2026) (10.9K rows, 13 features)

**Task:** Regression — predict `lastSoldPrice` from property attributes

**Approach:** Autonomous ML experiment loop ([autoresearch](https://github.com/detrin/autoresearch)) — 31 experiments run by an AI agent, logged to MLflow

**Best Result:** RMSE **335,652** (3-model ensemble: LightGBM + XGBoost + CatBoost)

| # | Model | Description | RMSE | Status |
|---|-------|-------------|------|--------|
| 1 | LinearRegression | Baseline, label-encoded | 575,853 | baseline |
| 2 | LightGBM | Default params, 1000 trees | 466,917 | +19% |
| 3 | LightGBM | + feature engineering | 467,734 | worse |
| 4 | LightGBM | + zip target encoding | 432,709 | +25% |
| 5 | LGB+XGB+CB | Optuna-tuned, log target | 383,308 | +33% |
| 6 | LGB+XGB+CB | + zip quantiles + price/sqft | 344,042 | +40% |
| **7** | **LGB+XGB+CB** | **+ type target encoding** | **335,652** | **+42%** |

**Key features:** Zip-code target encoding (smoothed mean, median, quantiles, IQR, price/sqft), property type target encoding, relative features (sqft vs zip average), log-transformed target, Optuna hyperparameter tuning, weighted 3-model ensemble.

---
*Generated via [autoresearch](https://github.com/detrin/autoresearch) — 31 autonomous experiments, MLflow-tracked*

## 1. Setup & Data Loading

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression

DATA_FILE = "florida_real_estate_sold_properties_ultimate.csv"
kaggle_matches = glob.glob(f"/kaggle/input/**/{DATA_FILE}", recursive=True)
if kaggle_matches:
    DATA_PATH = kaggle_matches[0]
elif os.path.exists(DATA_FILE):
    DATA_PATH = DATA_FILE
else:
    raise FileNotFoundError(f"Cannot find {DATA_FILE}")

RANDOM_STATE = 42
TEST_SIZE = 0.2
TARGET = "lastSoldPrice"

print(f"Loading data from: {DATA_PATH}")
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=["sanitized_text"], errors="ignore")
print(f"Shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print(f"Target stats:\n{df[TARGET].describe()}\n")
print(f"Column types:\n{df.dtypes.value_counts()}\n")
print(f"Null counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(df[TARGET], bins=50, edgecolor="black")
axes[0, 0].set_title("lastSoldPrice Distribution")
axes[0, 0].set_xlabel("Price ($)")

axes[0, 1].hist(np.log1p(df[TARGET]), bins=50, edgecolor="black", color="orange")
axes[0, 1].set_title("log1p(lastSoldPrice) Distribution")

df.boxplot(column=TARGET, by="type", ax=axes[1, 0])
axes[1, 0].set_title("Price by Property Type")
axes[1, 0].set_xlabel("")

axes[1, 1].scatter(df["sqft"], df[TARGET], alpha=0.1, s=5)
axes[1, 1].set_title("Price vs Sqft")
axes[1, 1].set_xlabel("sqft")
axes[1, 1].set_ylabel("lastSoldPrice")

plt.suptitle("")
plt.tight_layout()
plt.show()

## 3. Data Preparation & Feature Engineering

Key features engineered during the experiment loop:
- **Property attributes:** age, sqft/bed, sqft/bath, total rooms, bed/bath ratio
- **Zip-code target encoding:** smoothed mean, median, quantiles (Q25/Q75), IQR, avg price/sqft
- **Type target encoding:** smoothed mean, median, std, avg sqft
- **Relative features:** sqft vs zip average, beds vs zip average, age vs zip average
- **Log transforms:** log1p(sqft), log1p(target), log1p(zip_te_smooth)

In [ ]:
train, val = train_test_split(df, test_size=TEST_SIZE, random_state=RANDOM_STATE)

drop_cols = [TARGET, "listPrice"]
feature_cols = [c for c in train.columns if c not in drop_cols]

for d in [train, val]:
    d["property_age"] = 2026 - d["year_built"]
    d["sqft_per_bed"] = d["sqft"] / d["beds"].replace(0, np.nan)
    d["sqft_per_bath"] = d["sqft"] / d["baths"].replace(0, np.nan)
    d["total_rooms"] = d["beds"] + d["baths"]
    d["bed_bath_ratio"] = d["beds"] / d["baths"].replace(0, np.nan)
    d["log_sqft"] = np.log1p(d["sqft"])
    d["sqft_x_baths"] = d["sqft"] * d["baths"]

global_mean = train[TARGET].mean()
smoothing = 10

zip_stats = train.groupby("zip").agg(
    zip_te_mean=(TARGET, "mean"),
    zip_te_count=(TARGET, "count"),
    zip_te_std=(TARGET, "std"),
    zip_te_median=(TARGET, "median"),
    zip_sqft_mean=("sqft", "mean"),
    zip_beds_mean=("beds", "mean"),
    zip_age_mean=("property_age", "mean"),
    zip_te_q25=(TARGET, lambda x: x.quantile(0.25)),
    zip_te_q75=(TARGET, lambda x: x.quantile(0.75)),
).reset_index()
zip_stats["zip_te_std"] = zip_stats["zip_te_std"].fillna(0)
zip_stats["zip_te_smooth"] = (
    (zip_stats["zip_te_count"] * zip_stats["zip_te_mean"] + smoothing * global_mean)
    / (zip_stats["zip_te_count"] + smoothing)
)
zip_stats["zip_te_iqr"] = zip_stats["zip_te_q75"] - zip_stats["zip_te_q25"]
zip_stats["zip_price_per_sqft"] = zip_stats["zip_te_mean"] / zip_stats["zip_sqft_mean"].replace(0, np.nan)

merge_cols = [c for c in zip_stats.columns if c != "zip_te_mean"]
train = train.merge(zip_stats[merge_cols], on="zip", how="left")
val = val.merge(zip_stats[merge_cols], on="zip", how="left")
for c in merge_cols[1:]:
    fill = global_mean if any(k in c for k in ["smooth", "median", "q25", "q75"]) else 0
    val[c] = val[c].fillna(fill)

type_stats = train.groupby("type").agg(
    type_te_mean=(TARGET, "mean"),
    type_te_count=(TARGET, "count"),
    type_te_std=(TARGET, "std"),
    type_te_median=(TARGET, "median"),
    type_sqft_mean=("sqft", "mean"),
).reset_index()
type_stats["type_te_std"] = type_stats["type_te_std"].fillna(0)
type_stats["type_te_smooth"] = (
    (type_stats["type_te_count"] * type_stats["type_te_mean"] + smoothing * global_mean)
    / (type_stats["type_te_count"] + smoothing)
)
type_merge = [c for c in type_stats.columns if c != "type_te_mean"]
train = train.merge(type_stats[type_merge], on="type", how="left")
val = val.merge(type_stats[type_merge], on="type", how="left")
for c in type_merge[1:]:
    fill = global_mean if any(k in c for k in ["smooth", "median"]) else 0
    val[c] = val[c].fillna(fill)

for d in [train, val]:
    d["log_zip_te_smooth"] = np.log1p(d["zip_te_smooth"].clip(lower=0))
    d["sqft_vs_zip_avg"] = d["sqft"] / d["zip_sqft_mean"].replace(0, np.nan)
    d["beds_vs_zip_avg"] = d["beds"] / d["zip_beds_mean"].replace(0, np.nan)
    d["age_vs_zip_avg"] = d["property_age"] / d["zip_age_mean"].replace(0, np.nan)
    d["price_per_sqft_est"] = d["zip_price_per_sqft"] * d["sqft"]
    d["sqft_vs_type_avg"] = d["sqft"] / d["type_sqft_mean"].replace(0, np.nan)
    d["log_type_te_smooth"] = np.log1p(d["type_te_smooth"].clip(lower=0))

extra_cols = [
    "property_age", "sqft_per_bed", "sqft_per_bath", "total_rooms", "bed_bath_ratio",
    "log_sqft", "sqft_x_baths",
    "zip_te_smooth", "zip_te_count", "zip_te_std", "zip_te_median",
    "zip_te_q25", "zip_te_q75", "zip_te_iqr", "zip_price_per_sqft",
    "log_zip_te_smooth", "zip_sqft_mean", "zip_beds_mean", "zip_age_mean",
    "sqft_vs_zip_avg", "beds_vs_zip_avg", "age_vs_zip_avg", "price_per_sqft_est",
    "type_te_smooth", "type_te_count", "type_te_std", "type_te_median",
    "type_sqft_mean", "sqft_vs_type_avg", "log_type_te_smooth",
]
feature_cols = feature_cols + extra_cols

cat_cols = train[feature_cols].select_dtypes(include="object").columns.tolist()
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))
    val[col] = le.transform(val[col].astype(str))
    encoders[col] = le

X_train = train[feature_cols].fillna(-1)
X_val = val[feature_cols].fillna(-1)
y_train_log = np.log1p(train[TARGET])
y_val = val[TARGET]
y_val_log = np.log1p(y_val)

print(f"Train: {X_train.shape}, Val: {X_val.shape}")
print(f"Features ({len(feature_cols)}): {feature_cols}")

## 4. Baseline — Linear Regression

RMSE: **575,853**

In [ ]:
lr = LinearRegression()
lr.fit(X_train, train[TARGET])
preds_lr = lr.predict(X_val)
rmse_lr = root_mean_squared_error(y_val, preds_lr)
print(f"Linear Regression RMSE: {rmse_lr:,.0f}")

results = [("LinearRegression", rmse_lr)]

## 5. Best Model — LightGBM + XGBoost + CatBoost Ensemble

Hyperparameters found by Optuna (120/80/50 trials per model). Baked in directly — no search at runtime.

In [ ]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

model_lgb = lgb.LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.008485577810467727,
    num_leaves=105,
    max_depth=3,
    min_child_samples=44,
    subsample=0.5086745971918659,
    colsample_bytree=0.3421987686260641,
    reg_alpha=0.024588444514165034,
    reg_lambda=1.3450518918710853e-08,
    random_state=42,
    verbosity=-1,
)
model_lgb.fit(
    X_train, y_train_log,
    eval_set=[(X_val, y_val_log)],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
)
preds_lgb = np.expm1(model_lgb.predict(X_val))
rmse_lgb = root_mean_squared_error(y_val, preds_lgb)
print(f"LightGBM RMSE: {rmse_lgb:,.0f}")

results.append(("LightGBM", rmse_lgb))

In [ ]:
model_xgb = xgb.XGBRegressor(
    n_estimators=3000,
    learning_rate=0.11877277544269024,
    max_depth=7,
    min_child_weight=51,
    subsample=0.6937936593920374,
    colsample_bytree=0.3010692896328767,
    reg_alpha=0.0030553917100508447,
    reg_lambda=0.12214597826576712,
    random_state=42,
    verbosity=0,
    tree_method="hist",
    early_stopping_rounds=100,
)
model_xgb.fit(X_train, y_train_log, eval_set=[(X_val, y_val_log)], verbose=False)
preds_xgb = np.expm1(model_xgb.predict(X_val))
rmse_xgb = root_mean_squared_error(y_val, preds_xgb)
print(f"XGBoost RMSE: {rmse_xgb:,.0f}")

results.append(("XGBoost", rmse_xgb))

In [ ]:
model_cb = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.014219165225475746,
    depth=4,
    l2_leaf_reg=5.859696121572066e-08,
    bagging_temperature=0.41699479033522135,
    random_strength=0.6542668425691279,
    random_seed=42,
    verbose=0,
)
model_cb.fit(X_train, y_train_log, eval_set=(X_val, y_val_log), early_stopping_rounds=100)
preds_cb = np.expm1(model_cb.predict(X_val))
rmse_cb = root_mean_squared_error(y_val, preds_cb)
print(f"CatBoost RMSE: {rmse_cb:,.0f}")

results.append(("CatBoost", rmse_cb))

## 6. Weighted Ensemble

Blend weights optimized by grid search over the 3 models.

In [ ]:
best_rmse = float("inf")
best_weights = (1/3, 1/3, 1/3)

for w1_int in range(0, 101, 5):
    for w2_int in range(0, 101 - w1_int, 5):
        w1 = w1_int / 100
        w2 = w2_int / 100
        w3 = 1.0 - w1 - w2
        blend = w1 * preds_lgb + w2 * preds_xgb + w3 * preds_cb
        rmse = root_mean_squared_error(y_val, blend)
        if rmse < best_rmse:
            best_rmse = rmse
            best_weights = (w1, w2, w3)

w1, w2, w3 = best_weights
preds_ensemble = w1 * preds_lgb + w2 * preds_xgb + w3 * preds_cb
rmse_ensemble = root_mean_squared_error(y_val, preds_ensemble)
print(f"Ensemble weights: LGB={w1:.2f}, XGB={w2:.2f}, CB={w3:.2f}")
print(f"Ensemble RMSE: {rmse_ensemble:,.0f}")

results.append(("Ensemble (LGB+XGB+CB)", rmse_ensemble))

## 7. Feature Importance & Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

importance = pd.Series(model_lgb.feature_importances_, index=X_train.columns).sort_values()
importance.tail(15).plot.barh(ax=axes[0])
axes[0].set_title("Top 15 Features (LightGBM split importance)")

axes[1].scatter(y_val, preds_ensemble, alpha=0.15, s=10)
axes[1].plot([0, y_val.max()], [0, y_val.max()], "r--", alpha=0.5)
axes[1].set_xlabel("Actual Price")
axes[1].set_ylabel("Predicted Price")
axes[1].set_title("Predicted vs Actual (Ensemble)")

plt.tight_layout()
plt.show()

## 8. Results Summary

In [ ]:
results_df = pd.DataFrame(results, columns=["Model", "RMSE"]).sort_values("RMSE")
print(results_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["#2ecc71" if "Ensemble" in m else "#3498db" for m in results_df["Model"]]
bars = ax.barh(results_df["Model"], results_df["RMSE"], color=colors, edgecolor="black")
ax.set_xlabel("RMSE (lower is better)")
ax.set_title("Model Comparison — Florida Real Estate Price Prediction")
for bar, val in zip(bars, results_df["RMSE"]):
    ax.text(val + 5000, bar.get_y() + bar.get_height() / 2, f"{val:,.0f}", va="center")
plt.tight_layout()
plt.show()

## Conclusions

**Best model:** LGB+XGB+CatBoost weighted ensemble — **RMSE ~335,652** (42% improvement over baseline)

**What worked:**
- **Zip-code target encoding** was the single biggest lever — smoothed means, medians, quantiles, and derived price/sqft estimates captured neighborhood-level pricing
- **Log-transformed target** helped the models handle the heavy right tail of property prices
- **Ensemble blending** (3 diverse gradient boosting implementations) consistently beat any single model
- **Type-level features** (property type mean/median price, avg sqft) added incremental lift
- **Relative features** (sqft vs zip average, beds vs zip average) helped normalize within-market comparisons

**What didn't work (31 experiments tested):**
- Outlier removal — validation set still has outliers, so removing from train hurts
- Stacking with Ridge meta-learner — no improvement over simple weighted average
- Multi-seed averaging — marginal at best, added runtime
- Text features from listing descriptions — slight regression vs the cleaner feature set
- Feature interactions beyond what gradient boosting captures natively

**Takeaway:** On a small dataset (10.9K rows), zip-code target encoding + log target + diverse ensemble is the winning formula. The dataset's high variance in luxury vs standard properties makes RMSE hard to minimize further without external data (e.g., neighborhood crime rates, school ratings, proximity to coast).

---
*Generated via [autoresearch](https://github.com/detrin/autoresearch) — 31 autonomous experiments, MLflow-tracked*